# ReFactX: New Features Interactive Test

This notebook tests the three new features:
1. **Count Branches Tool** – `count_branches(<prefix>)` inline tool
2. **Sentinel for Exhausted Relations** – emits `no further records>` when all objects are generated
3. **Thinking Cache Reset** – resets duplicate cache after `</think>

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import time
from transformers.generation.logits_process import LogitsProcessorList
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

import refactx
from refactx.count_branches import CountBranchesLogitsProcessor

## Configuration

In [ ]:
MODEL = 'Qwen/Qwen3-0.6B'
INDEX = '../indexes/simple_index.txt.gz'

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

## Load Model and Index

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, device_map='auto')
streamer = TextStreamer(tokenizer)
print(f'Model loaded: {MODEL}')

In [ ]:
index = refactx.load_index(INDEX, tokenizer=tokenizer)
print(f'Index loaded: {len(index)} triples')

In [ ]:
refactx.patch_model(model)

## Helper Function

In [ ]:
def ask(question, logits_processor=None, max_new_tokens=800, sentinel=True):
    """Ask a question and return the output text and generated facts."""
    prompted = [refactx.apply_prompt_template(tokenizer, question=question)]
    inputs = tokenizer(text=prompted, return_tensors='pt', padding=True, padding_side='right')
    inputs = inputs.to(model.device)

n    if logits_processor is None:
        logits_processor = refactx.get_constrained_logits_processor(
            tokenizer, index, num_beams=1, num_batches=1, sentinel=sentinel)

    model.eval()
    start = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            logits_processor=logits_processor,
            max_new_tokens=max_new_tokens,
            streamer=streamer,
            do_sample=False,
            temperature=None,
            top_k=None,
            num_beams=1,
            num_return_sequences=1,
            use_cache=True,
            top_p=None,
            min_p=None,
        )
    elapsed = time.time() - start

    _from = len(inputs.input_ids[0])
    text = tokenizer.decode(out[0][_from:])
    facts = refactx.get_constrained_states()[0][0].generated_triples
    facts_str = [tokenizer.decode(t) for t in facts]

    print(f'\n--- Facts ({len(facts_str)}) ---')
    for i, f in enumerate(facts_str):
        print(f'  {i}: {f}')
    print(f'Elapsed: {elapsed:.2f}s')

    return text, facts_str

## Feat 1: Count Branches Tool

The `CountBranchesLogitsProcessor` detects `count_branches(<prefix>)` in the generated text
and forces emission of ` = <count>` tokens.

In [ ]:
constrained_proc = refactx.get_constrained_logits_processor(
    tokenizer, index, num_beams=1, num_batches=1, return_list=False)
count_branches_proc = CountBranchesLogitsProcessor(tokenizer, index)
logits_processor_list = LogitsProcessorList([constrained_proc, count_branches_proc])

text, facts = ask(
    'How many countries share a border with Spain?',
    logits_processor=logits_processor_list)

## Feat 2: Sentinel for Exhausted Relations

When `sentinel=True`, exhausted subject-relations emit `no further records>`
instead of being pruned.

In [ ]:
text, facts = ask(
    'Which countries share a border with Spain?',
    sentinel=True,
    max_new_tokens=1200)

## Feat 3: Thinking Cache Reset

When a model uses `<think>` blocks, the duplicate cache is reset after `</think>`,
allowing triple repetition in post-thinking output.

In [ ]:
text, facts = ask(
    'Is Johnny Depp older than Brad Pitt?',
    sentinel=True,
    max_new_tokens=1200)

## Interactive Testing

Edit the question below and run to test.

In [ ]:
# Change this question to test interactively
MY_QUESTION = 'Who is the CEO of Tesla?'

text, facts = ask(MY_QUESTION, sentinel=True)